# **02 웹 검색 에이전트 구현하기**

### 학습 내용
1. Tavily Search API 이해 및 설정
2. TavilySearch 도구 사용법
3. 웹 검색 결과 처리
4. create_agent와 Tavily Search 통합

## 0. 환경 설정

- OpenAI API Key 발급: https://platform.openai.com/api-keys
- Tavily API Key 발급: https://app.tavily.com/home

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

if os.environ.get("OPENAI_API_KEY"):
    print("OPENAI API Key가 설정되었습니다.")

if os.getenv("TAVILY_API_KEY"):
    print("TAVILY API Key가 설정되었습니다.")

OPENAI API Key가 설정되었습니다.
TAVILY API Key가 설정되었습니다.


## 1. Tavily Search API 소개

**Tavily Search API**는 AI 에이전트(LLM)를 위해 특별히 설계된 검색 엔진입니다.

### 주요 특징

- **실시간 검색**: 최신 정보를 빠르게 검색
- **AI 최적화**: LLM이 이해하기 쉬운 형식으로 결과 제공
- **정확한 결과**: 신뢰할 수 있는 소스에서 팩트 기반 정보 제공
- **무료 티어**: 월 1,000회 무료 검색 제공

## 2. TavilySearch 도구 기본 사용법

[TavilySearch](https://reference.langchain.com/python/langchain-tavily/tavily_search/TavilySearch) 도구를 직접 사용하여 웹 검색을 수행해봅시다.

In [2]:
from langchain_tavily import TavilySearch

# 기본 설정으로 TavilySearch 도구 생성
search_tool = TavilySearch(
    max_results=3,  # 최대 검색 결과 수
    topic="general",  # 검색 카테고리: "general", "news", "finance"
)

print(f"도구 이름: {search_tool.name}")
print(f"도구 설명: {search_tool.description}")

도구 이름: tavily_search
도구 설명: A search engine optimized for comprehensive, accurate, and trusted results. Useful for when you need to answer questions about current events. It not only retrieves URLs and snippets, but offers advanced search depths, domain management, time range filters, and image search, this tool delivers real-time, accurate, and citation-backed results.Input should be a search query.


In [3]:
result = search_tool.invoke({"query": "LangChain이란 무엇인가요?"})

print("\n=== 검색 결과 ===")
print(result)


=== 검색 결과 ===
{'query': 'LangChain이란 무엇인가요?', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://aws.amazon.com/ko/what-is/langchain', 'title': 'LangChain이란 무엇인가요?', 'content': '검색\n\n# LangChain이란 무엇인가요?\n\n## LangChain이란 무엇인가요?\n\nLangChain은 대규모 언어 모델(LLM)을 기반으로 애플리케이션을 구축하기 위한 오픈 소스 프레임워크입니다. LLM은 대량의 데이터로 사전 훈련된 대규모 딥 러닝 모델로서, 질문에 답하거나 텍스트 기반 프롬프트에서 이미지를 생성하는 등 사용자 쿼리에 대한 응답을 생성할 수 있습니다. LangChain은 모델이 생성하는 정보의 맞춤화, 정확성 및 관련성을 개선하기 위한 도구와 추상화 기능을 제공합니다. 예를 들어 개발자는 LangChain 구성 요소를 사용하여 새 프롬프트 체인을 구축하거나 기존 템플릿을 맞춤화할 수 있습니다. LangChain에는 LLM이 재훈련 없이 새 데이터 세트에 액세스할 수 있도록 하는 구성 요소도 포함되어 있습니다.\n\n대규모 언어 모델(LLM)에 대해 읽어보기\n\n## LangChain이 중요한 이유는 무엇인가요?', 'score': 0.9409691, 'raw_content': None, 'id': '44f410-00'}, {'url': 'https://cloud.google.com/use-cases/langchain?hl=ko', 'title': 'LangChain이란 무엇인가요? 예시 및 정의', 'content': '# LangChain이란 무엇인가요?\n\nLangChain은 대규모 언어 모델(LLM)로 애플리케이션을 더 쉽게 빌드할 수 있도록 지원하는 오픈소스 조정 프레임워크입니다. LLM을 다양한 데이터 소스에 연결하는 도구와 구성요소를

In [4]:
result["results"]

[{'url': 'https://aws.amazon.com/ko/what-is/langchain',
  'title': 'LangChain이란 무엇인가요?',
  'content': '검색\n\n# LangChain이란 무엇인가요?\n\n## LangChain이란 무엇인가요?\n\nLangChain은 대규모 언어 모델(LLM)을 기반으로 애플리케이션을 구축하기 위한 오픈 소스 프레임워크입니다. LLM은 대량의 데이터로 사전 훈련된 대규모 딥 러닝 모델로서, 질문에 답하거나 텍스트 기반 프롬프트에서 이미지를 생성하는 등 사용자 쿼리에 대한 응답을 생성할 수 있습니다. LangChain은 모델이 생성하는 정보의 맞춤화, 정확성 및 관련성을 개선하기 위한 도구와 추상화 기능을 제공합니다. 예를 들어 개발자는 LangChain 구성 요소를 사용하여 새 프롬프트 체인을 구축하거나 기존 템플릿을 맞춤화할 수 있습니다. LangChain에는 LLM이 재훈련 없이 새 데이터 세트에 액세스할 수 있도록 하는 구성 요소도 포함되어 있습니다.\n\n대규모 언어 모델(LLM)에 대해 읽어보기\n\n## LangChain이 중요한 이유는 무엇인가요?',
  'score': 0.9409691,
  'raw_content': None,
  'id': '44f410-00'},
 {'url': 'https://cloud.google.com/use-cases/langchain?hl=ko',
  'title': 'LangChain이란 무엇인가요? 예시 및 정의',
  'content': '# LangChain이란 무엇인가요?\n\nLangChain은 대규모 언어 모델(LLM)로 애플리케이션을 더 쉽게 빌드할 수 있도록 지원하는 오픈소스 조정 프레임워크입니다. LLM을 다양한 데이터 소스에 연결하는 도구와 구성요소를 제공하여 복잡한 다단계 워크플로를 만들 수 있습니다.\n\nLangChain은 Python 및 JavaScript 라이브러리로 제공되며, 개발자가 LLM을 외부 데이터 및 계산에 연

### 주요 파라미터

| 파라미터 | 설명 | 기본값 |
|---------|------|-------|
| `max_results` | 반환할 최대 검색 결과 수 | 5 |
| `topic` | 검색 카테고리 ("general", "news", "finance") | "general" |
| `include_answer` | 질문에 대한 직접 답변 포함 여부 | False |
| `include_raw_content` | 전체 HTML 콘텐츠 포함 여부 | False |
| `include_images` | 관련 이미지 포함 여부 | False |
| `search_depth` | 검색 깊이 ("basic", "advanced") | "basic" |
| `include_domains` | 포함할 도메인 리스트 | None |
| `exclude_domains` | 제외할 도메인 리스트 | None |
| `time_range` | 시간 범위 ("day", "week", "month", "year") | None |

## 3. 특정 도메인으로 검색 제한하기

특정 웹사이트만 검색하거나 특정 사이트를 제외할 수 있습니다.

In [5]:
# Wikipedia만 검색하기
wiki_search = TavilySearch(
    max_results=3,
    include_domains=["wikipedia.org"],  # Wikipedia만 검색
)

result = wiki_search.invoke({"query": "인공지능"})

print("\n=== Wikipedia 검색 결과 ===")
print(result)


=== Wikipedia 검색 결과 ===
{'query': '인공지능', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://ko.wikipedia.org/wiki/%EC%9D%B8%EA%B3%B5%EC%A7%80%EB%8A%A5', 'title': '인공지능 - 위키백과, 우리 모두의 백과사전', 'content': '인공지능(人工智能, 영어: artificial intelligence, AI)은 인간의 학습능력, 추론능력, 지각능력을 인공적으로 구현하려는 컴퓨터 과학의 세부분야 중 하나이다. 정보공학 분야에 있어 하나의 인프라 기술이기도 하다. 인간을 포함한 동물이 갖고 있는 지능 즉, 자연 지능(natural intelligence)과는 다른 개념이다.\n\n인간의 지능을 모방한 기능을 갖춘 컴퓨터 시스템이며, 인간의 지능을 기계 등에 인공적으로 시연(구현)한 것이다. 일반적으로 범용 컴퓨터에 적용한다고 가정한다. 이 용어는 또한 그와 같은 지능을 만들 수 있는 방법론이나 실현 가능성 등을 연구하는 과학 기술 분야를 지칭하기도 한다.\n\n## 강인공지능과 약인공지능 [...] 약인공지능(weak AI)은 사진에서 물체를 찾거나 소리를 듣고 상황을 파악하는 것과 같이 기존에 인간은 쉽게 해결할 수 있으나 컴퓨터로 처리하기에는 어려웠던 각종 문제를 컴퓨터로 수행하게 만드는데 중점을 두고 있다. 한참 막연한 인간 지능을 목표로 하기보다는 더 현실적으로 실용적인 목표를 가지고 개발되고 있는 인공지능이라고 할 수 있으며, 일반적인 지능을 가진 무언가라기보다는 특정한 문제를 해결하는 도구로써 활용된다. [...] 강인공지능(strong AI) 또는 인공 일반 지능(artificial general intelligence, AGI)은 인간처럼 실제로 사고하여 문제를 해결할 수 있는 "일반 지능"을 인공적으로 구현하려는 시도이다. 오늘날 이 분야의 연구는 주로

In [6]:
# 특정 도메인 제외하기
filtered_search = TavilySearch(
    max_results=3,
    exclude_domains=["reddit.com", "twitter.com", "blog.naver.com", "tistory.com"],
)

result = filtered_search.invoke({"query": "파이썬 프로그래밍 팁"})

print("\n=== 필터링된 검색 결과 ===")
print(result)


=== 필터링된 검색 결과 ===
{'query': '파이썬 프로그래밍 팁', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.lucypark.kr/courses/tips/introduction-to-python.html', 'title': 'Introduction to Python (on Windows) — Courses', 'content': "1. 에러 메세지를 잘 읽자 (read, not see). 흔히들 시험 볼 때 문제 속에 답이 있다고 하는데, 프로그래밍할 때는 에러 메세지 속에 이유가 있다. 진짜다.\n2. 구글링!\n    주의: 구글링을 한다고 무조건 좋은 레퍼런스가 찾아지는 것은 아니다\n    보통은 공식 문서 (official documents) / API가 가장 좋은 레퍼런스 (ex: docs.python.org <- 웹문서의 도메인을 잘 보라.)\n3. Learn how to program, rather than just accomplishing the task\n    프로그래밍을 배우는 것과 복/붙하는 것은 다르다 (참고: Teach yourself programming in ten years)\n    일단 프로그램이 작동하게 하는데 성공했다면, 왜 작동했는지를 이해하자.\n4. Follow coding conventions. 많은 똑똑한 사람들이 선택한 방법에는 이유가 있다.\n    PEP-8 [...] > There are two hard things in computer science: cache invalidation, naming things, and off-by-one errors.\n>\n> — Jeff Atwood (@codinghorror) August 31, 2014\n\n### 윈도우 프로그래머를 위한 팁\n\n1. 윈도우에서 자주 속썩이는 것들\n\n   1. 인코딩이 골치. UTF8 쓰는

## 4. create_agent와 Tavily Search 통합

이제 Tavily Search를 에이전트와 통합하여 더 지능적인 검색 에이전트를 만들어봅시다.

In [7]:
from langchain.agents import create_agent

# Tavily Search 도구 생성
tavily_tool = TavilySearch(
    max_results=5,
    topic="general",
)

# 검색 에이전트 생성
search_agent = create_agent(
    model="gpt-5.4-mini",
    tools=[tavily_tool],
    system_prompt="""당신은 유용한 연구 보조 AI입니다.
    Tavily 검색 도구를 사용하여 정확하고 최신 정보를 찾아주세요.""",
)

In [8]:
for chunk_msg, metadata in search_agent.stream({"messages": [{"role": "user", "content": "2026년의 AI 트렌드는?"}]}, stream_mode="messages"):
    print(chunk_msg.content, end="", flush=True)

{"query": "2026 AI trends latest 2026 predictions enterprise generative AI agentic AI multimodal AI standards regulation", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.linkedin.com/pulse/ai-2026-agents-multimodal-models-regulatory-milestones-sanjay-gupta-aiybc", "title": "AI in 2026: Agents, Multimodal Models, and Regulatory Milestones", "content": "Show code\n\nEnterprise Adoption Trends\n\nThe combination of smarter models and new platforms is driving a Cambrian explosion in AI use across businesses. Surveys and forecasts highlight the scale and speed of change. Cisco reports that, by 2027, 80% of executives believe their firm’s survival hinges on agentic AI, and most expect over half their workers to collaborate with AI agents within two years. Gartner similarly predicted that ~40% of enterprise applications will include task-specific AI agents by end of 2026 (up from <5% in 2025), and that AI agents could generate ~$450B in software re

## 5. 실전 연습 1: 뉴스 분석 에이전트

최신 뉴스를 검색하고 분석하는 에이전트를 만들어봅시다.

In [9]:
# 뉴스 검색용 도구
news_search = TavilySearch(
    max_results=5,
    topic="news",
    start_date="2025-01-01"
)

from datetime import datetime
current_date = datetime.now().strftime("%Y-%m-%d")

# 뉴스 분석 에이전트
news_agent = create_agent(
    model="gpt-5.4-mini",
    tools=[news_search],
    system_prompt=f"""당신은 뉴스 분석 보조 AI입니다.
    시사 이슈에 대해 질문을 받으면:
    1. 해당 주제에 대한 최신 뉴스 검색
    2. 출처 제공

    분석은 객관적이고 사실에 기반해야 합니다.
    현재 날짜 : {current_date}""",
)

In [17]:
for chunk_msg, metadata in news_agent.stream({"messages": [{"role": "user", "content": "오디세이 영화에 대한 모든 것"}]}, stream_mode="messages"):
    print(chunk_msg.content, end="", flush=True)

{"query": "오디세이 영화 'The Odyssey' latest news Christopher Nolan 2026 cast release date OR OR OR OR", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.empireonline.com/movies/news/the-odyssey-firstlooks-at-tom-holland-anne-hathaway-and-mia-goth-and-tom-holland-in-christopher-nolans-epic-cno", "title": "Christopher Nolan’s The Odyssey: Tom Holland, Anne Hathaway, Mia Goth First Looks", "content": "Read Empire’s full, super-sized The Odyssey cover story – venturing to Christopher Nolan’s LA offices for a world-exclusive interview about his jaw-droppingly ambitious new epic, and getting the first word from Matt Damon on Odysseus’ wild journey – in the January 2026 issue of Empire, on sale Thursday 20 November. Pre-order a copy online here. The Odyssey comes to UK cinemas from 17 July 2026.\n\nStay tuned to empireonline.com for more The Odyssey exclusives, coming soon.\n\nJust so you know, we may receive a commission or other compensation from the l

## 6. 실전 연습 2: 학술 연구 보조 에이전트

특정 도메인(학술 사이트)만 검색하는 연구 보조 에이전트를 만들어봅시다.

In [11]:
# 학술 검색용 도구
academic_search = TavilySearch(
    max_results=5,
    include_domains=[
        "arxiv.org",
        "scholar.google.com",
        "wikipedia.org",
        "nature.com",
        "sciencedirect.com"
    ],
    search_depth="advanced",
)

# 연구 보조 에이전트
research_agent = create_agent(
    model="gpt-5.4-mini",
    tools=[academic_search],
    system_prompt="""당신은 학술 연구 보조 AI입니다.
    사용자가 신뢰할 수 있는 출처에서 학술 정보를 찾도록 도와주세요.
    다음에 중점을 두세요:
    - 과학적 정확성
    - 동료 평가된 출처
    - 최신 연구 결과
    - 복잡한 주제에 대한 명확한 설명

    추가 읽기를 위해 항상 출처 URL을 제공하세요.""",
)

In [12]:
from langchain_core.messages import AIMessageChunk

for chunk_msg, metadata in research_agent.stream({"messages": [{"role": "user", "content": "2025년 ~ 2026년의 프롬프트 엔지니어링의 연구 동향을 알려주세요."}]}, stream_mode="messages"):
    if type(chunk_msg) == AIMessageChunk:
        print(chunk_msg.content, end="", flush=True)

아래는 **2025~2026년 프롬프트 엔지니어링(prompt engineering) 연구 동향**을 학술 문헌 흐름에 맞춰 정리한 내용입니다.  
요약하면, **“프롬프트를 잘 쓰는 기술”에서 “컨텍스트를 설계하고, 프롬프트를 자동 최적화하며, 에이전트 시스템 전체를 조율하는 기술”로 중심이 이동**하고 있습니다.

---

## 1) 핵심 변화: 프롬프트 엔지니어링 → 컨텍스트 엔지니어링 → 에이전트 설계

2025~2026년 가장 두드러진 변화는, 연구 초점이 단일 프롬프트 작성에서 벗어나 **컨텍스트 관리(context management)** 와 **에이전트 오케스트레이션(agent orchestration)** 으로 확장된 점입니다.

- 과거: “어떻게 질문을 잘 던질 것인가?”
- 현재: “모델이 행동할 때 어떤 정보, 기억, 도구, 규칙을 어떤 구조로 제공할 것인가?”

이 흐름은 2026년 문헌에서 **context engineering** 이란 용어로 강하게 나타납니다.  
컨텍스트 엔지니어링은 프롬프트 문장 자체보다, **모델이 순간적으로 접하는 상태(state) 전체를 설계**하는 방향입니다.

### 관련 출처
- Context Engineering: From Prompts to Corporate Multi-Agent Architecture (arXiv, 2026)  
  https://arxiv.org/pdf/2603.09619
- A Survey of Context Engineering for Large Language Models (arXiv, 2025로 인용됨)  
  https://arxiv.org/search/?query=A+Survey+of+Context+Engineering+for+Large+Language+Models&searchtype=all

---

## 2) 자동 프롬프트 최적화(APO)가 본격적인 연구 주류로 진입

2025년에는 **Automatic Prompt Optimization(APO)** 가 독립적인

In [13]:
from langchain_core.messages import AIMessage

full_message = ""

for chunk_msg, metadata in research_agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "코딩 특화 AI Agent에 대한 연구 자료를 찾아주세요."
            }
        ]
    },
    stream_mode="messages",
):
    print(chunk_msg.content, end="", flush=True)

    # AIMessage만 저장
    if isinstance(chunk_msg, AIMessage):
        if not chunk_msg.tool_calls:
            full_message += chunk_msg.content

{"query": "software engineering AI agent code generation study large language models", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://arxiv.org/html/2508.17343v3", "title": "Agentic AI for Software: thoughts from Software Engineering community", "content": "# Agentic AI for Software: thoughts from Software Engineering community\n\n###### Abstract\n\nAI agents have recently shown significant promise in software engineering. Much public attention has been transfixed on the topic of code generation from Large Language Models (LLMs) via a prompt. However, software engineering is much more than programming, and AI agents go far beyond instructions given by a prompt. [...] A broad structure of Large Language Model (LLM) agents for software has emerged in the past 1-2 years. An agent is autonomous and makes (informed) decisions on its own. It may be informed by invoking various external tools. In the case of LLM agents for software - the most effective

In [14]:
from IPython.display import Markdown
display(Markdown(full_message))

코딩 특화 AI Agent(= LLM 기반 코드 생성/수정/디버깅 에이전트)에 대한 연구 자료를 핵심 위주로 정리해드리겠습니다. 아래 자료들은 주로 **서베이(survey)**, **벤치마크**, **시스템/프레임워크**, **실증 연구**로 나뉩니다.

## 1) 가장 먼저 읽기 좋은 서베이
1. **A Survey on Code Generation with LLM-based Agents**  
   - 코드 생성 에이전트의 개념, 단일/다중 에이전트 구조, SDLC(소프트웨어 개발 생명주기) 적용, 평가 벤치마크를 폭넓게 정리합니다.  
   - 링크: https://arxiv.org/html/2508.00083v1

2. **AI Agentic Programming: A Survey of Techniques, Challenges, and Opportunities**  
   - 코딩 에이전트가 계획(planning), 도구 사용(tool use), 디버깅, 버전 관리와 상호작용하는 방식 중심으로 정리한 최신 서베이입니다.  
   - 링크: https://arxiv.org/html/2508.11126v2

3. **Software Development Life Cycle Perspective: A Survey of Benchmarks for CodeLLMs and Agents**  
   - 코딩 에이전트 평가를 위한 벤치마크들을 SDLC 관점에서 체계적으로 분류합니다.  
   - 링크: https://arxiv.org/html/2505.05283v1

## 2) 코딩 에이전트의 대표 벤치마크
4. **SWE-bench**  
   - 실제 GitHub 이슈를 해결하는 능력을 평가하는 대표 벤치마크입니다.  
   - 코딩 에이전트 논문 대부분이 이 벤치마크를 사용합니다.  
   - 참고: https://www.swebench.com/  
   - 논문 검색: https://arxiv.org/search/?query=SWE-bench&searchtype=all

5. **ProjDevBench: Benchmarking AI Coding Agents on End-to-End Project Development**  
   - 단일 함수 생성이 아니라, 프로젝트 전체를 end-to-end로 구현하는 능력을 평가합니다.  
   - 링크: https://arxiv.org/html/2602.01655v1

6. **SWE-EVO: Benchmarking Coding Agents in Long-Horizon Software Evolution**  
   - 장기적인 코드 수정, 릴리스 노트 반영, 다중 파일 변경 등 “진짜 소프트웨어 진화”에 가까운 과제를 평가합니다.  
   - 링크: https://arxiv.org/html/2512.18470v5

## 3) 대표적인 코딩 에이전트 시스템
7. **SWE-agent**  
   - 에이전트가 실제 컴퓨터/터미널과 상호작용하며 소프트웨어 이슈를 해결하는 대표적 접근입니다.  
   - 링크: https://arxiv.org/html/2405.15793v1

8. **AutoCodeRover**  
   - GitHub 이슈 해결을 위한 자동화 에이전트 계열 연구로, 검색·로컬라이제이션·수정 전략이 중요합니다.  
   - 링크: https://arxiv.org/search/?query=AutoCodeRover&searchtype=all

9. **AgentCoder**  
   - programmer / test designer / test executor처럼 역할을 분리한 다중 에이전트 프레임워크입니다.  
   - 관련 소개가 포함된 서베이: https://arxiv.org/html/2406.00515v2

10. **OpenDevin / Devin 계열**  
   - “AI software engineer”를 지향하는 실사용형 에이전트로, 연구와 산업 모두에서 큰 관심을 받았습니다.  
   - 관련 서베이 언급: https://arxiv.org/html/2406.00515v2

## 4) 최근 연구 동향을 이해하는 데 유용한 자료
11. **Agentic AI for Software: thoughts from Software Engineering community**  
   - 소프트웨어 공학 관점에서 에이전트가 왜 중요한지, 어떤 구조가 효과적인지 정리한 논의성 자료입니다.  
   - 링크: https://arxiv.org/html/2508.17343v3

12. **LLM-Based Agentic Systems for Software Engineering: Challenges and Opportunities**  
   - 멀티에이전트, 협업, 비용, 인간-에이전트 조정 등 향후 과제를 다룹니다.  
   - 링크: https://arxiv.org/html/2601.09822v2

## 5) 현재 연구에서 반복적으로 나오는 핵심 쟁점
- **단순 코드 생성 vs. 자율적 문제 해결**  
  - 이제는 한 번에 코드만 생성하는 것이 아니라, 요구사항 파악 → 계획 → 코드 수정 → 테스트 → 디버깅까지 수행하는지가 중요합니다.
- **도구 사용(tool use)**  
  - 터미널, 컴파일러, 디버거, IDE, Git, 테스트 실행기 등을 어떻게 통합하느냐가 성능을 좌우합니다.
- **평가 문제**  
  - 함수 단위 정답률보다, 실제 저장소 수준의 이슈 해결 능력, 장기 작업, 유지보수성까지 평가하는 벤치마크가 중요해졌습니다.
- **신뢰성/안전성**  
  - 환각(hallucination), 무한 루프, 잘못된 수정, 테스트 통과는 했지만 유지보수성이 떨어지는 코드 등 문제가 큽니다.

## 6) 추천 읽기 순서
1. **A Survey on Code Generation with LLM-based Agents**  
2. **SWE-bench 관련 자료**  
3. **SWE-agent / AutoCodeRover / AgentCoder**  
4. **AI Agentic Programming survey**  
5. **ProjDevBench, SWE-EVO 같은 최신 벤치마크**

원하시면 다음 단계로 이어서:
- **논문별 핵심 기여/한계 비교표**
- **한국어로 된 1페이지 요약**
- **“코딩 에이전트” 연구를 위한 참고문헌 리스트(BibTeX 스타일)**  
형태로 정리해드릴 수 있습니다.

### 📖 과제 1: 도메인 특화 웹 검색 에이전트 만들기

위에서 학습한 내용을 바탕으로, **특정 도메인에 특화된 웹 검색 에이전트**를 만들어보세요.

- 출처를 포함한 정확한 답변 제공

In [15]:
# CODE HERE

<details>
<summary>참고 예시 (클릭하여 펼치기)</summary>

### 레시피 검색 에이전트 예시

```python
from langchain_tavily import TavilySearch
from langchain.agents import create_agent

# 레시피 전문 검색 도구
recipe_search = TavilySearch(
    max_results=5,
    include_domains=[
        "allrecipes.com",
        "foodnetwork.com", 
        "tasty.co",
        "bbcgoodfood.com",
        "10000recipe.com"  # 만개의 레시피
    ],
)

# 레시피 에이전트
recipe_agent = create_agent(
    model="gpt-5.4-mini",
    tools=[recipe_search],
    system_prompt="""당신은 친절한 요리 도우미 AI입니다.
    사용자가 요리 레시피를 요청하면:
    1. 신뢰할 수 있는 레시피 사이트에서 최신 정보 검색
    2. 재료, 조리 시간, 난이도를 포함하여 요약
    3. 초보자도 이해하기 쉽게 설명
    4. 출처 URL을 반드시 제공하여 자세한 레시피 확인 가능하도록 함
    
    항상 맛있고 건강한 요리를 만들 수 있도록 도와주세요!""",
)

# 테스트
for chunk_msg, metadata in recipe_agent.stream(
    {"messages": [{"role": "user", "content": "초보자를 위한 간단한 파스타 레시피 알려줘"}]},
    stream_mode="messages"
):
    print(chunk_msg.content, end="", flush=True)
```

</details>

---
### 참고 자료

- [Tavily Search Integration](https://docs.langchain.com/oss/python/integrations/tools/tavily_search)
- [Tavily API Documentation](https://docs.tavily.com/documentation/api-reference/endpoint/search)
- [LangChain Agents](https://docs.langchain.com/oss/python/langchain/agents)
- [LangChain Tools](https://docs.langchain.com/oss/python/langchain/tools)